In [ ]:
# 🔧 STEP 0: FORCE AGGRESSIVE SEGMENTATION SETTINGS\n# Run this cell FIRST to apply very aggressive settings for maximum segmentation\n\nprint(\"🔧 Applying ULTRA-AGGRESSIVE Segmentation Settings\")\nprint(\"=\" * 60)\n\ndef force_aggressive_segmentation():\n    \"\"\"Apply the most aggressive settings possible for segmentation\"\"\"\n    config = OmegaConf.load('../configs/mesh_segmentation.yaml')\n    \n    print(\"🔥 Setting MAXIMUM aggression parameters:\")\n    \n    # Ultra-low thresholds for maximum segments\n    config.sam.model_type = 'sam3'\n    config.sam.sam.text_prompt = \"individual part\"\n    config.sam.sam.threshold = 0.1  # Ultra-low\n    config.sam.sam.mask_threshold = 0.1  # Ultra-low\n    config.sam.sam.engine_config.points_per_side = 128  # Maximum points\n    print(f\"   • Thresholds: {config.sam.sam.threshold} (ultra-low for more segments)\")\n    print(f\"   • Points per side: {config.sam.sam.engine_config.points_per_side} (maximum)\")\n    \n    # Ultra-aggressive mesh processing\n    config.sam_mesh.min_area = 32  # Tiny minimum area\n    config.sam_mesh.repartition_lambda = 1  # Maximum segments\n    config.sam_mesh.smoothing_iterations = 4  # Minimal smoothing\n    config.sam_mesh.connections_bin_threshold_percentage = 0.01  # Ultra-low\n    config.sam_mesh.smoothing_threshold_percentage_size = 0.001  # Keep tiny segments\n    config.sam_mesh.smoothing_threshold_percentage_area = 0.001  # Keep tiny areas\n    print(f\"   • Min area: {config.sam_mesh.min_area} (ultra-small)\")\n    print(f\"   • Lambda: {config.sam_mesh.repartition_lambda} (max segments)\")\n    print(f\"   • Smoothing: {config.sam_mesh.smoothing_iterations} (minimal)\")\n    \n    # Try multiple rendering modes\n    config.sam_mesh.use_modes = ['norms', 'sdf', 'matte']  # All modes\n    print(f\"   • Modes: {config.sam_mesh.use_modes} (all available)\")\n    \n    # Force cache refresh\n    config.cache_overwrite = True\n    \n    # Save the ultra-aggressive config\n    OmegaConf.save(config, '../configs/mesh_segmentation.yaml')\n    print(\"✅ Ultra-aggressive settings saved!\")\n    \n    return config\n\n# Apply the settings\nultra_config = force_aggressive_segmentation()\n\nprint(\"\\n🎯 Alternative Approach - Try SAM2 if SAM3 fails:\")\ndef try_sam2_fallback():\n    \"\"\"Switch to SAM2 with aggressive settings\"\"\"\n    config = OmegaConf.load('../configs/mesh_segmentation.yaml')\n    config.sam.model_type = 'sam2'  # Switch to SAM2\n    config.sam.sam2.engine_config.points_per_side = 64\n    config.sam.sam2.engine_config.pred_iou_thresh = 0.3  # Lower threshold\n    config.sam.sam2.engine_config.stability_score_thresh = 0.3  # Lower threshold\n    config.cache_overwrite = True\n    OmegaConf.save(config, '../configs/mesh_segmentation.yaml')\n    print(\"🔄 Switched to SAM2 with aggressive settings\")\n    return config\n\nprint(\"\\n💡 If segmentation still fails after running the main cells:\")\nprint(\"   1. Run: try_sam2_fallback()\")\nprint(\"   2. Then re-run the segmentation cells\")\nprint(\"   3. Or try different text prompts manually\")\n\nprint(\"\\n🚀 Now run the other cells - segmentation should be MUCH more aggressive!\")"

# SAM3 Mesh Segmentation - Automated

## 🚀 **Just Run All Cells!**

This notebook is fully automated:
1. **Run Cell 1**: Loads backend and finds your mesh files
2. **Run Cell 2**: Automatically applies optimal settings
3. **Run Cell 3**: Runs segmentation with best parameters

**Features:**
- ✅ Auto-detects mesh files
- ✅ Optimized SAM3 parameters
- ✅ Multiple text prompts tried automatically
- ✅ Fallback to SAM2 if needed


In [ ]:
# 🔧 STEP 1: BACKEND SETUP & AUTO-DETECTION
# This cell loads everything and finds your mesh files automatically

%load_ext autoreload
%autoreload 2

import os
import trimesh
import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree
from omegaconf import OmegaConf
from samesh.data.loaders import *
from samesh.models.sam_mesh import *

print("🚀 SAM3 Automated Mesh Segmentation")
print("=" * 50)

def get_first_mesh_or_combined(mesh_or_scene) -> trimesh.Trimesh:
    """Extract single mesh from Trimesh or Scene object"""
    if isinstance(mesh_or_scene, trimesh.Trimesh):
        return mesh_or_scene
    elif isinstance(mesh_or_scene, trimesh.Scene):
        if len(mesh_or_scene.geometry) == 0:
            raise ValueError("No geometry found in the scene.")
        combined = trimesh.util.concatenate(tuple(mesh_or_scene.geometry.values()))
        return combined
    else:
        raise TypeError("Loaded object is neither a Trimesh nor a Scene.")

def merge_vertices_by_distance_blender_style(mesh: trimesh.Trimesh, threshold: float = 1e-4) -> trimesh.Trimesh:
    """Merge vertices by distance using KDTree (Blender-style)"""
    mesh = mesh.copy()
    verts = mesh.vertices
    kdtree = cKDTree(verts)
    groups = kdtree.query_ball_tree(kdtree, threshold)

    vert_map = np.arange(len(verts))
    for i, group in enumerate(groups):
        min_idx = min(group)
        for j in group:
            vert_map[j] = min_idx

    unique_indices = np.unique(vert_map)
    index_mapping = {old_idx: new_idx for new_idx, old_idx in enumerate(unique_indices)}
    
    new_vertices = verts[unique_indices]
    new_faces = np.array([[index_mapping[vert_map[v]] for v in face] for face in mesh.faces])
    
    return trimesh.Trimesh(vertices=new_vertices, faces=new_faces)

def auto_optimize_config(text_prompt):
    """Automatically optimize configuration for best segmentation"""
    config = OmegaConf.load('../configs/mesh_segmentation.yaml')
    
    # Force SAM3 with optimized settings
    config.sam.model_type = 'sam3'
    config.sam.sam.text_prompt = text_prompt
    config.sam.sam.threshold = 0.2  # Low for more segments
    config.sam.sam.mask_threshold = 0.2
    config.sam.sam.engine_config.points_per_side = 64
    
    # Optimize mesh processing
    config.sam_mesh.min_area = 128
    config.sam_mesh.repartition_lambda = 2
    config.sam_mesh.smoothing_iterations = 16
    config.sam_mesh.connections_bin_threshold_percentage = 0.05
    
    # Force cache refresh
    config.cache_overwrite = True
    
    return config

def segment_mesh_auto(file_path, text_prompt):
    """Automated mesh segmentation with error handling"""
    print(f"🎯 Segmenting: {Path(file_path).name}")
    print(f"📝 Prompt: '{text_prompt}'")
    print("-" * 40)
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Mesh file not found: {file_path}")
    
    # Get optimized config
    config = auto_optimize_config(text_prompt)
    
    # Load and preprocess mesh
    print("📊 Loading and preprocessing mesh...")
    original_mesh = trimesh.load(file_path)
    processed_mesh = get_first_mesh_or_combined(original_mesh)
    processed_mesh = merge_vertices_by_distance_blender_style(processed_mesh, 0.0001)
    
    print(f"   Processed: {len(processed_mesh.vertices)} vertices, {len(processed_mesh.faces)} faces")
    
    # Create temporary file
    output_dir = Path(config.output)
    output_dir.mkdir(parents=True, exist_ok=True)
    temp_file = output_dir / "temp_mesh.obj"
    processed_mesh.export(temp_file)
    
    # Run segmentation
    print(f"🚀 Running SAM3 segmentation...")
    try:
        segmented_mesh = segment_mesh(temp_file, config, visualize=True)
        print("✅ Segmentation completed!")
        print(f"📁 Outputs saved to: {config.output}")
        return segmented_mesh, True
        
    except Exception as e:
        print(f"❌ SAM3 failed: {e}")
        return None, False
    finally:
        if temp_file.exists():
            temp_file.unlink()

def find_available_meshes():
    """Find available mesh files automatically"""
    search_paths = [
        "PATH",
        "PATHnloads/",
        "./"
    ]
    
    extensions = ["*.glb", "*.obj", "*.ply", "*.stl"]
    found_meshes = []
    
    for search_path in search_paths:
        path = Path(search_path)
        if path.exists():
            for ext in extensions:
                found_meshes.extend(path.glob(ext))
    
    return sorted(set(found_meshes))

# Auto-detect mesh files
print("🔍 Auto-detecting mesh files...")
available_meshes = find_available_meshes()

if available_meshes:
    print(f"✅ Found {len(available_meshes)} mesh files:")
    for i, mesh in enumerate(available_meshes[:3], 1):  # Show first 3
        print(f"   {i}. {mesh.name}")
    if len(available_meshes) > 3:
        print(f"   ... and {len(available_meshes) - 3} more")
    
    # Select the first mesh automatically
    selected_mesh = str(available_meshes[0])
    print(f"\n🎯 Auto-selected: {Path(selected_mesh).name}")
else:
    print("❌ No mesh files found!")
    print("Please add mesh files to one of these locations:")
    print("   • PATH")
    print("   • PATHnloads/")
    selected_mesh = None

print("\n✅ Backend loaded successfully!")

In [ ]:
# 🎯 STEP 2: AUTOMATED SEGMENTATION WITH SMART PROMPTS
# This cell tries multiple prompts automatically to find the best segmentation

if selected_mesh is None:
    print("❌ No mesh file available. Please add mesh files and re-run the previous cell.")
else:
    print("🤖 Starting Automated Segmentation")
    print("=" * 50)
    
    # Smart prompt selection based on filename
    filename = Path(selected_mesh).stem.lower()
    
    # Auto-select best prompts based on filename
    if any(word in filename for word in ['chair', 'seat', 'furniture', 'table']):
        prompts_to_try = [
            "furniture component",
            "chair part", 
            "different section",
            "distinct object part"
        ]
        print("🪑 Detected furniture - using furniture-specific prompts")
    elif any(word in filename for word in ['mech', 'gear', 'engine', 'machine']):
        prompts_to_try = [
            "mechanical component",
            "separate part",
            "machine element",
            "distinct object part"
        ]
        print("⚙️ Detected mechanical object - using mechanical prompts")
    else:
        prompts_to_try = [
            "distinct object part",
            "separate component", 
            "different section",
            "individual element"
        ]
        print("🔍 Using general-purpose prompts")
    
    print(f"\n📝 Will try {len(prompts_to_try)} different prompts:")
    for i, prompt in enumerate(prompts_to_try, 1):
        print(f"   {i}. '{prompt}'")
    
    # Try each prompt until we get a good result
    best_result = None
    successful_prompt = None
    
    for i, prompt in enumerate(prompts_to_try, 1):
        print(f"\n🔄 Attempt {i}/{len(prompts_to_try)}: '{prompt}'")
        print("-" * 30)
        
        try:
            result, success = segment_mesh_auto(selected_mesh, prompt)
            
            if success and result is not None:
                print(f"✅ SUCCESS with prompt: '{prompt}'")
                best_result = result
                successful_prompt = prompt
                break
            else:
                print(f"⚠️ Prompt '{prompt}' didn't work well, trying next...")
                
        except Exception as e:
            print(f"❌ Error with prompt '{prompt}': {e}")
            continue
    
    # Results summary
    print("\n" + "=" * 50)
    if best_result is not None:
        print(f"🎉 SEGMENTATION SUCCESSFUL!")
        print(f"📝 Best prompt: '{successful_prompt}'")
        print(f"📁 File: {Path(selected_mesh).name}")
        print(f"\n📺 Displaying result...")
        
        # Show the result
        try:
            best_result.show()
        except:
            print("Note: 3D viewer may not work in all environments")
            
    else:
        print("❌ All prompts failed. Trying fallback to SAM2...")
        
        # Fallback to SAM2
        try:
            config = OmegaConf.load('../configs/mesh_segmentation.yaml')
            config.sam.model_type = 'sam2'  # Switch to SAM2
            config.cache_overwrite = True
            
            print("🔄 Trying SAM2 fallback...")
            fallback_result = segment_mesh_auto(selected_mesh, "object part")
            
            if fallback_result[1]:  # If successful
                print("✅ SAM2 fallback successful!")
                best_result = fallback_result[0]
                best_result.show()
            else:
                print("❌ SAM2 fallback also failed")
                
        except Exception as e:
            print(f"❌ Fallback failed: {e}")
            print("\n💡 Troubleshooting suggestions:")
            print("   • Check if mesh file is valid")
            print("   • Try a simpler/lower-poly mesh")
            print("   • Ensure mesh has no holes or errors")
            print("   • Check GPU memory availability")

In [ ]:
# 📊 STEP 3: RESULTS SUMMARY & NEXT STEPS
# This cell shows you what happened and what to do next

print("📊 Segmentation Results Summary")
print("=" * 50)

if 'best_result' in locals() and best_result is not None:
    print("✅ STATUS: Segmentation completed successfully!")
    print(f"📁 Processed file: {Path(selected_mesh).name}")
    
    if 'successful_prompt' in locals():
        print(f"📝 Winning prompt: '{successful_prompt}'")
    
    # Check output directory
    output_dir = Path('PATHntation_output')
    if output_dir.exists():
        output_files = list(output_dir.glob('*'))
        print(f"\n📁 Output files generated: {len(output_files)}")
        print(f"   Location: {output_dir}")
        
        # Show some key files
        key_files = []
        for f in output_files:
            if 'segmented' in f.name or 'face2label' in f.name:
                key_files.append(f)
        
        if key_files:
            print("   Key files:")
            for f in key_files[:3]:  # Show first 3
                print(f"     • {f.name}")
    
    print("\n🎨 Visual Result:")
    print("   • The mesh should now show multiple colors")
    print("   • Each color represents a different segment/part")
    print("   • If you see multiple colors, segmentation worked!")
    
    print("\n🔄 To try different settings:")
    print("   • Re-run this notebook with a different mesh")
    print("   • Modify the prompts_to_try list in Step 2")
    print("   • Check the output files for detailed results")
    
else:
    print("❌ STATUS: Segmentation failed")
    print("\n🔧 Troubleshooting steps:")
    print("   1. Check if your mesh file is valid (try opening in Blender/MeshLab)")
    print("   2. Ensure the mesh is manifold (no holes, proper geometry)")
    print("   3. Try with a simpler mesh first")
    print("   4. Check GPU memory (SAM3 needs ~4GB+ VRAM)")
    print("   5. Try running individual cells to isolate the issue")
    
    print("\n💡 Quick fixes to try:")
    print("   • Restart the notebook kernel")
    print("   • Clear the cache directory")
    print("   • Try with a different mesh file")

print("\n🎉 Notebook execution complete!")
print("\nIf you see multiple colors in your mesh, segmentation worked perfectly! 🎨")

In [ ]:
best_result.show()

In [ ]:
# 🎨 STEP 4: DISPLAY OUTPUT MESH\n# This cell shows the segmented mesh with multiple visualization options\n\nif 'best_result' in locals() and best_result is not None:\n    print(\"🎨 Displaying Segmented Mesh\")\n    print(\"=\" * 50)\n    \n    try:\n        # Show the main result\n        print(\"📺 Opening 3D viewer...\")\n        best_result.show()\n        \n        # Additional visualization options\n        print(\"\\n🔍 Mesh Information:\")\n        print(f\"   Vertices: {len(best_result.vertices):,}\")\n        print(f\"   Faces: {len(best_result.faces):,}\")\n        \n        # Check if mesh has colors (indicating segmentation worked)\n        if hasattr(best_result.visual, 'face_colors') and best_result.visual.face_colors is not None:\n            unique_colors = np.unique(best_result.visual.face_colors.view(np.void), return_counts=True)\n            num_segments = len(unique_colors[0])\n            print(f\"   Segments: {num_segments} different colored regions\")\n            \n            if num_segments > 1:\n                print(\"   ✅ Segmentation successful - multiple colors detected!\")\n            else:\n                print(\"   ⚠️ Only one color detected - segmentation may not have worked\")\n        else:\n            print(\"   ⚠️ No color information found\")\n        \n        # Save options\n        print(\"\\n💾 Save Options:\")\n        print(\"   # Uncomment any of these lines to save the result:\")\n        print(\"   # best_result.export('segmented_mesh.glb')\")\n        print(\"   # best_result.export('segmented_mesh.obj')\")\n        print(\"   # best_result.export('segmented_mesh.ply')\")\n        \n        # Alternative viewers\n        print(\"\\n🔧 Alternative Viewing Options:\")\n        print(\"   If the 3D viewer doesn't work, try:\")\n        print(\"   • Check the output directory for saved files\")\n        print(\"   • Open the .glb/.obj files in Blender or MeshLab\")\n        print(\"   • Use the visualization images in the output folder\")\n        \n    except Exception as e:\n        print(f\"❌ Error displaying mesh: {e}\")\n        print(\"\\n💡 Alternative options:\")\n        print(\"   • Check the output directory for saved files\")\n        print(\"   • The segmentation may have worked even if display failed\")\n        \nelse:\n    print(\"❌ No segmented mesh available to display\")\n    print(\"Please run the previous cells first to generate a segmented mesh.\")\n\n# Quick save function\ndef save_result(filename=\"segmented_mesh.glb\"):\n    \"\"\"Quick function to save the segmented mesh\"\"\"\n    if 'best_result' in locals() and best_result is not None:\n        best_result.export(filename)\n        print(f\"✅ Saved mesh as: {filename}\")\n    else:\n        print(\"❌ No mesh to save. Run segmentation first.\")\n\nprint(\"\\n💡 To save the mesh, run: save_result('my_mesh.glb')\")"

In [ ]:
# 🩺 DEBUGGING: Why Isn't Segmentation Working?\n# Run this cell if you're still getting single-color meshes\n\ndef debug_segmentation_pipeline():\n    \"\"\"Debug the segmentation pipeline to find issues\"\"\"\n    print(\"🩺 Segmentation Debugging\")\n    print(\"=\" * 50)\n    \n    # Check if we have a result\n    if 'best_result' not in locals() or best_result is None:\n        print(\"❌ No segmentation result found\")\n        return\n    \n    # Analyze the mesh colors in detail\n    print(\"🔍 Detailed Color Analysis:\")\n    if hasattr(best_result.visual, 'face_colors'):\n        face_colors = best_result.visual.face_colors\n        if face_colors is not None:\n            print(f\"   Face colors shape: {face_colors.shape}\")\n            print(f\"   Color data type: {face_colors.dtype}\")\n            \n            # Check unique colors\n            unique_colors = np.unique(face_colors.reshape(-1, face_colors.shape[-1]), axis=0)\n            print(f\"   Unique colors found: {len(unique_colors)}\")\n            \n            if len(unique_colors) <= 3:  # Show all colors if few\n                print(\"   Color values:\")\n                for i, color in enumerate(unique_colors):\n                    print(f\"     Color {i+1}: {color}\")\n            \n            # Check if colors are too similar\n            if len(unique_colors) > 1:\n                color_diffs = np.diff(unique_colors, axis=0)\n                avg_diff = np.mean(np.abs(color_diffs))\n                print(f\"   Average color difference: {avg_diff:.3f}\")\n                \n                if avg_diff < 10:  # Very similar colors\n                    print(\"   ⚠️ Colors are very similar - may appear as single color\")\n        else:\n            print(\"   ❌ No face colors found\")\n    else:\n        print(\"   ❌ No visual.face_colors attribute\")\n    \n    # Check output files for clues\n    output_dir = Path('PATHntation_output')\n    if output_dir.exists():\n        print(f\"\\n📁 Output Directory Analysis:\")\n        files = list(output_dir.glob('*'))\n        print(f\"   Total files: {len(files)}\")\n        \n        # Look for face2label file\n        face2label_files = [f for f in files if 'face2label' in f.name]\n        if face2label_files:\n            print(f\"   Face2label files: {len(face2label_files)}\")\n            # Try to load and analyze\n            try:\n                import json\n                with open(face2label_files[0], 'r') as f:\n                    face2label = json.load(f)\n                \n                labels = list(face2label.values())\n                unique_labels = set(labels)\n                print(f\"   Unique labels in face2label: {len(unique_labels)}\")\n                print(f\"   Label values: {sorted(unique_labels)[:10]}...\")  # Show first 10\n                \n                if len(unique_labels) <= 1:\n                    print(\"   ❌ PROBLEM: Only one label found - segmentation failed\")\n                else:\n                    print(f\"   ✅ Multiple labels found - segmentation worked at face level\")\n                    \n            except Exception as e:\n                print(f\"   ❌ Error reading face2label: {e}\")\n        else:\n            print(\"   ❌ No face2label files found\")\n    \n    print(\"\\n💡 Troubleshooting Suggestions:\")\n    print(\"   1. Try even more aggressive settings (lower thresholds)\")\n    print(\"   2. Switch to SAM2: try_sam2_fallback()\")\n    print(\"   3. Try different text prompts: 'chair leg', 'surface', 'edge'\")\n    print(\"   4. Check if mesh is too simple (not enough geometric variation)\")\n    print(\"   5. Try a different mesh file\")\n\n# Auto-run if we have a result\nif 'best_result' in locals():\n    debug_segmentation_pipeline()\nelse:\n    print(\"🔄 Run segmentation first, then come back to this cell for debugging\")\n\n# Quick test function\ndef quick_test_different_prompts():\n    \"\"\"Test different prompts quickly\"\"\"\n    if 'selected_mesh' not in locals():\n        print(\"❌ No mesh selected\")\n        return\n        \n    test_prompts = [\n        \"chair leg\",\n        \"furniture part\", \n        \"surface region\",\n        \"geometric feature\",\n        \"object boundary\",\n        \"distinct area\"\n    ]\n    \n    print(\"🧪 Quick Prompt Testing:\")\n    for prompt in test_prompts:\n        print(f\"\\n🔄 Testing: '{prompt}'\")\n        try:\n            result, success = segment_mesh_auto(selected_mesh, prompt)\n            if success:\n                print(f\"✅ SUCCESS with '{prompt}'!\")\n                return result, prompt\n            else:\n                print(f\"❌ Failed with '{prompt}'\")\n        except Exception as e:\n            print(f\"❌ Error with '{prompt}': {e}\")\n    \n    print(\"\\n❌ All test prompts failed\")\n    return None, None\n\nprint(\"\\n🧪 To test different prompts: quick_test_different_prompts()\")"

In [ ]:
# 💾 STEP 5: AUTO-SAVE TO GLB & DISPLAY\n# This cell automatically saves the mesh to GLB format and displays it\n\nif 'best_result' in locals() and best_result is not None:\n    print(\"💾 Auto-Save and Display\")\n    print(\"=\" * 40)\n    \n    try:\n        # Auto-generate filename based on original mesh\n        if 'selected_mesh' in locals():\n            output_filename = f\"segmented_{Path(selected_mesh).stem}.glb\"\n        else:\n            output_filename = \"segmented_mesh.glb\"\n        \n        # Save the mesh\n        print(f\"💾 Saving mesh as: {output_filename}\")\n        best_result.export(output_filename)\n        print(f\"✅ Mesh saved successfully!\")\n        \n        # Show file info\n        saved_path = Path(output_filename).absolute()\n        print(f\"📁 Saved to: {saved_path}\")\n        print(f\"📊 File size: {saved_path.stat().st_size / 1024:.1f} KB\")\n        \n        # Display the mesh\n        print(\"\\n📺 Displaying saved mesh...\")\n        best_result.show()\n        \n        print(f\"\\n🎉 Complete! Your segmented mesh is saved as '{output_filename}'\")\n        \n    except Exception as e:\n        print(f\"❌ Error saving mesh: {e}\")\n        print(\"\\n🔄 Trying to display without saving...\")\n        try:\n            best_result.show()\n        except:\n            print(\"❌ Display also failed\")\n            \nelse:\n    print(\"❌ No mesh available to save\")\n    print(\"Run the previous segmentation cells first!\")"